In [92]:
import pandas as pd
from tqdm import tqdm

In [137]:
df_ground_truth = pd.read_csv("data/ground_truth_v3.csv") #-data.csv")

In [138]:
df_ground_truth.shape

(607, 2)

In [139]:
df_ground_truth.head()

,question,document
0,"Quick homemade applesauce recipe with apples and cinnamon, about 25 minutes?",2
1,"What’s an easy apple dessert I can make with apples, oats, brown sugar, and butter in about an hour?",5
2,"Quick 20 minute dessert with apples, cinnamon, and butter?",9
3,What can I make with apples and puff pastry in about an hour?,10
4,I have apples and butter on hand and about an hour — what’s an easy apple crisp recipe?,11


In [140]:
ground_truth = df_ground_truth.to_dict(orient="records")


In [141]:
from ingest import load_data, build_index

documents = load_data()
index = build_index(documents)

In [142]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [143]:
boost = {'total_time': 1.0, 'ingredients': 1.0}

index.search(
    "Any quick apple dessert or snack recipe with just apples, sugar, cinnamon, and water?",
    num_results=5,
    boost_dict=boost
)

[{'id': '60',
  'recipe_name': 'Delicious Cinnamon Baked Apples',
  'total_time': '1 hrs',
  'servings': '6',
  'ingredients': '1 teaspoon butter, 2 tablespoons brown sugar, 3 teaspoons vanilla sugar, 3 teaspoons ground cinnamon, 1 teaspoon ground nutmeg, 6 large apples - peeled, cored, and sliced, 3 ½ tablespoons water',
  'directions': 'Preheat the oven to 350 degrees F (175 degrees C). Grease a large baking dish with butter.\nMix brown sugar, vanilla sugar, cinnamon, and nutmeg in a small bowl. Layer about 1/3 of the apples in the prepared baking dish; sprinkle with 1/3 of the sugar mixture. Repeat layers twice more.\nBake in preheated oven for 30 minutes. Pour water over apples and continue baking until tender, about 15 minutes more.',
  'nutrition': 'Total Fat 1g 2%, Saturated Fat 1g 3%, Cholesterol 2mg 1%, Sodium 9mg 0%, Total Carbohydrate 37g 13%, Dietary Fiber 6g 21%, Total Sugars 27g, Protein 1g, Vitamin C 10mg 49%, Calcium 29mg 2%, Iron 0mg 2%, Potassium 240mg 5%'},
 {'id': '

In [144]:
def text_search(query):
    boost_dict = {'total_time': 1.0, 'ingredients': 1.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [145]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(int(d["id"]) == int(doc_id)))

    return relevance

In [146]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [147]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 607/607 [00:04<00:00, 135.48it/s]


In [148]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [149]:
hit_rate(relevance_total)

0.8797364085667215

In [150]:
len(relevance_total)

607

In [151]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [152]:
mrr(relevance_total)

0.6613124656781981

In [153]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [154]:
evaluate(ground_truth, text_search)

100%|██████████| 607/607 [00:04<00:00, 138.95it/s]


{'hit_rate': 0.8797364085667215, 'mrr': 0.6613124656781981}

In [155]:
df_validation = df_ground_truth.sample(frac=0.8, random_state=42)
df_test = df_ground_truth.drop(df_validation.index)

gt_val = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

In [156]:
def search_boosts(query, recipe_name_boost, ingredients_boost, nutrition_boost):
    boost_dict = {
        "recipe_name": recipe_name_boost,
        "ingredients": ingredients_boost,
        "nutrition": nutrition_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [157]:
results = []

for recipe_name_boost in [0.5, 1.0, 2.0]:
    for ingredients_boost in [1.0, 2.0, 5.0, 10.0]:
        for nutrition_boost in [0.5, 1.0, 2.0]:
            print(f"Evaluating recipe_name_boost={recipe_name_boost}, ingredients_boost={ingredients_boost}, nutrition_boost={nutrition_boost}...")
            result = evaluate(
                gt_val,
                lambda query, recipe_name_boost=recipe_name_boost, ingredients_boost=ingredients_boost, nutrition_boost=nutrition_boost: search_boosts(
                    query,
                    recipe_name_boost,
                    ingredients_boost,
                    nutrition_boost
                )
            )

            results.append({
                "recipe_name": recipe_name_boost,
                "ingredients": ingredients_boost,
                "nutrition": nutrition_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating recipe_name_boost=0.5, ingredients_boost=1.0, nutrition_boost=0.5...


100%|██████████| 486/486 [00:03<00:00, 129.04it/s]


Evaluating recipe_name_boost=0.5, ingredients_boost=1.0, nutrition_boost=1.0...


100%|██████████| 486/486 [00:03<00:00, 138.97it/s]


Evaluating recipe_name_boost=0.5, ingredients_boost=1.0, nutrition_boost=2.0...


...

100%|██████████| 486/486 [00:03<00:00, 128.18it/s]


Evaluating recipe_name_boost=2.0, ingredients_boost=10.0, nutrition_boost=1.0...


100%|██████████| 486/486 [00:03<00:00, 140.61it/s]


Evaluating recipe_name_boost=2.0, ingredients_boost=10.0, nutrition_boost=2.0...


100%|██████████| 486/486 [00:03<00:00, 140.46it/s]


In [158]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,recipe_name,ingredients,nutrition,hit_rate,mrr
30,2.0,5.0,0.5,0.897119,0.748285
31,2.0,5.0,1.0,0.895062,0.746742
32,2.0,5.0,2.0,0.890947,0.742181
18,1.0,5.0,0.5,0.868313,0.716324
19,1.0,5.0,1.0,0.866255,0.714918
27,2.0,2.0,0.5,0.886831,0.713100
15,1.0,2.0,0.5,0.880658,0.708162
28,2.0,2.0,1.0,0.884774,0.704321
20,1.0,5.0,2.0,0.860082,0.703944
16,1.0,2.0,1.0,0.884774,0.700995


In [159]:
def text_search(query):
    boost_dict = {'recipe_name': 2.0, 'ingredients': 5.0, 'nutrition': 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [160]:
evaluate(gt_test, text_search)

100%|██████████| 121/121 [00:00<00:00, 145.71it/s]


{'hit_rate': 0.9256198347107438, 'mrr': 0.7367768595041323}

In [161]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [162]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [163]:
from evaluation_utils import RAGWithUsage

In [164]:
assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [165]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[str(doc_id)]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [166]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [167]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/607 [00:00<?, ?it/s]

In [170]:
df_results = pd.DataFrame(results)
df_results.head(1)

,question,answer_llm,answer_orig,document
0,"Quick homemade applesauce recipe with apples and cinnamon, about 25 minutes?","Yes — here’s a quick homemade applesauce recipe that takes about 25 minutes:\n\n**Sarah’s Homemade Applesauce**\n- **Total time:** 25 mins\n- **Servings:** 4\n\n**Ingredients:**\n- 4 apples, peeled, cored, and chopped\n- ¾ cup water\n- ¼ cup white sugar\n- ½ teaspoon ground cinnamon\n\n**Directions:**\n1. Combine the apples, water, sugar, and cinnamon in a saucepan.\n2. Cover and cook over medium heat until the apples are soft, about 15 to 20 minutes.\n3. Let the mixture cool.\n4. Mash with a fork or potato masher until it reaches the consistency you like.\n\n**Nutrition per serving:**\n- Calories not listed\n- Total Fat: 0g\n- Sodium: 3mg\n- Total Carbohydrate: 32g\n- Dietary Fiber: 4g\n- Total Sugars: 27g\n- Protein: 0g\n\nIf you want, I can also help make it less sweet or give you a no-sugar version.","{'id': '2', 'recipe_name': 'Sarah's Homemade Applesauce', 'total_time': '25 mins', 'servings': '4', 'ingredients': '4 apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon', 'directions': 'Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes. Allow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like. Photo by cookin'mama. cookin mama ', 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}",2


In [171]:
df_results.to_csv("data/rag_results_v3.csv", index=False)

In [172]:
assistant.total_cost()

1.2565417500000013

In [173]:
assistant.reset_usage()

In [174]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [177]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a user
2. The original suggested recipe from a database (ground truth data)
3. An recipe suggestion generated by an AI assistant

Your task is to decide if the AI recipe suggestion fulfills the user's request.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [178]:
aqa_judge_prompt = """
Question:
{question}

Original Recipe (ground truth):
{answer_orig}

AI Recipe:
{answer_llm}
""".strip()

In [179]:
answers = df_results.to_dict(orient="records")

In [180]:
rec = answers[0]
rec

{'question': 'Quick homemade applesauce recipe with apples and cinnamon, about 25 minutes?',
 'answer_llm': 'Yes — here’s a quick homemade applesauce recipe that takes about 25 minutes:\n\n**Sarah’s Homemade Applesauce**\n- **Total time:** 25 mins\n- **Servings:** 4\n\n**Ingredients:**\n- 4 apples, peeled, cored, and chopped\n- ¾ cup water\n- ¼ cup white sugar\n- ½ teaspoon ground cinnamon\n\n**Directions:**\n1. Combine the apples, water, sugar, and cinnamon in a saucepan.\n2. Cover and cook over medium heat until the apples are soft, about 15 to 20 minutes.\n3. Let the mixture cool.\n4. Mash with a fork or potato masher until it reaches the consistency you like.\n\n**Nutrition per serving:**\n- Calories not listed\n- Total Fat: 0g\n- Sodium: 3mg\n- Total Carbohydrate: 32g\n- Dietary Fiber: 4g\n- Total Sugars: 27g\n- Protein: 0g\n\nIf you want, I can also help make it less sweet or give you a no-sugar version.',
 'answer_orig': {'id': '2',
  'recipe_name': "Sarah's Homemade Applesauce"

In [181]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
Quick homemade applesauce recipe with apples and cinnamon, about 25 minutes?

Original Recipe (ground truth):
{'id': '2', 'recipe_name': "Sarah's Homemade Applesauce", 'total_time': '25 mins', 'servings': '4', 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon', 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n", 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

AI Recipe:
Yes — here’s a quick homemade applesauce recipe that takes about 25 minutes:

**Sarah’s Homemade Applesauce**
- **Total time:** 25 mins

In [182]:
from evaluation_utils import llm_structured_retry

In [183]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning="The AI recipe matches the ground truth recipe closely: same title, total time, servings, ingredients, and directions. It answers the user's request for a quick homemade applesauce recipe with apples and cinnamon in about 25 minutes. Minor omissions in the nutrition details do not affect correctness.", score='good')

In [184]:
from evaluation_utils import calc_price, calc_total_price

In [185]:
calc_price(usage)

{'input_cost': 0.00052275,
 'output_cost': 0.000324,
 'total_cost': 0.0008467500000000001}

In [186]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [187]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [188]:
judge_record(answers[0])

({'question': 'Quick homemade applesauce recipe with apples and cinnamon, about 25 minutes?',
  'document': 2,
  'score': 'good',
  'reasoning': "The AI recipe matches the ground truth closely: it gives a 25-minute homemade applesauce recipe with apples and cinnamon, includes the correct ingredients, time, servings, and directions. Minor omissions in nutrition details do not affect fulfillment of the user's request."},
 ResponseUsage(input_tokens=697, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=66, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=763))

In [189]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/607 [00:00<?, ?it/s]

In [190]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [191]:
calc_total_price(usages)


0.5626289999999999

In [192]:
df_eval = pd.DataFrame(evaluations)
df_eval.head()

,question,document,score,reasoning
0,"Quick homemade applesauce recipe with apples and cinnamon, about 25 minutes?",2,good,"The AI recipe matches the ground truth on the key requested elements: it is a quick homemade applesauce recipe, uses apples and cinnamon, has about 25 minutes total time, and includes the same ingredients and cooking/mashing directions. Minor differences in nutrition details do not affect correctness."
1,"What’s an easy apple dessert I can make with apples, oats, brown sugar, and butter in about an hour?",5,good,"The AI answer identifies the same dessert as the ground truth: Easy Apple Crisp with Oat Topping. It includes the key ingredients requested by the user (apples, oats, brown sugar, butter) and gives a preparation time of about an hour, matching the source. The directions are also consistent with the original recipe, with only minor omissions like exact measurements, which are acceptable for this task."
2,"Quick 20 minute dessert with apples, cinnamon, and butter?",9,good,"The AI recipe matches the user's request for a quick 20-minute dessert with apples, cinnamon, and butter. It includes those core ingredients and a skillet-cooked apple dessert. While it differs from the ground truth in ingredient amounts, adds nutmeg, uses brown sugar instead of white sugar, and changes the number of apples/servings, these are acceptable variations and do not change the core recipe concept or time frame."
3,What can I make with apples and puff pastry in about an hour?,10,good,"The user asked for something to make with apples and puff pastry in about an hour. The AI suggests a puff pastry apple strudel taking 40 minutes, which fits the time constraint and uses the requested ingredients. Although it is not the exact ground-truth recipe, it answers the request appropriately and even mentions Apple Turnovers as a close alternative."
4,I have apples and butter on hand and about an hour — what’s an easy apple crisp recipe?,11,good,"The AI recipe matches the ground truth recipe closely: same ingredients, oven temperature, baking dish, method, bake time, servings, and nutrition values. It answers the user's request for an easy apple crisp recipe using apples and butter in about an hour. Extra wording and offer to shorten are irrelevant but not incorrect."


In [193]:
df_eval.score.value_counts(normalize=True)

score
good    0.937397
bad     0.062603
Name: proportion, dtype: float64

In [73]:
pd.options.display.max_colwidth = 1000

In [87]:
df_eval[df_eval["score"] == "bad"].sample()

,question,document,score,reasoning
1068,"Got about an hour and I’ve got milk, eggs, sugar, butter, and maybe some coconut — what’s a simple dessert I can make with those?",957,bad,"The user asked for a simple dessert using milk, eggs, sugar, butter, and possibly coconut, with about an hour available. The ground truth recipe is Impossible Coconut Pie, which fits these ingredients and time, but the AI suggested a different coconut cake that requires additional pantry items like flour, baking powder, salt, and coconut milk, and it does not identify the simple pie/dessert implied by the source. Since it misses the key recipe and the ingredient match is poor, the answer does not fulfill the request."


In [88]:
df_results[df_results["document"] == 957]

,question,answer_llm,answer_orig,document
1068,"Got about an hour and I’ve got milk, eggs, sugar, butter, and maybe some coconut — what’s a simple dessert I can make with those?","A simple option from the recipes is **Coconut Coconut Milk Cake**. It takes **about 1 hour** total and uses **eggs, sugar, butter, and coconut** plus a few pantry staples.\n\n### What you’d need\n- 2 cups white sugar \n- 1 cup butter, softened \n- 4 eggs, separated \n- 1 cup unsweetened shredded coconut \n- 2 cups all-purpose flour \n- 2 tablespoons baking powder \n- 1 teaspoon salt \n- 1 can coconut milk \n\n### Quick method\n1. Preheat oven to **350°F (175°C)** and grease a **9x13-inch pan**.\n2. Beat **sugar, butter, and egg yolks** until smooth; mix in coconut.\n3. Mix **flour, baking powder, and salt** in another bowl.\n4. Stir the butter mixture into the dry ingredients, alternating with **coconut milk**, until just mixed.\n5. Beat the **egg whites** until soft peaks form, then gently fold them into the batter.\n6. Pour into the pan and bake for about **45 minutes**, until a toothpick comes out clean.\n\n### Nutrition\nPer serving: **15g fat, 26g carbs, 3g protein, 17...","{'id': '957', 'recipe_name': 'Impossible Coconut Pie', 'total_time': '55 mins', 'servings': '8', 'ingredients': '2 cups milk, ¾ cup white sugar, ½ cup baking mix, ¼ cup unsalted butter, softened, 4 large eggs, 1 ½ teaspoons vanilla extract, 1 cup flaked coconut (Optional)', 'directions': 'Preheat the oven to 350 degrees F (175 degrees C). Grease a 9-inch pie plate. Combine milk, sugar, baking mix, butter, eggs, and vanilla in blender; blend on low speed for 3 minutes. Pour into the prepared pie plate and let stand for 5 minutes. Sprinkle coconut over top. Bake in the preheated oven until a toothpick inserted into the center comes out clean, about 40 minutes. Serve warm.', 'nutrition': 'Total Fat 12g 16%, Saturated Fat 8g 38%, Cholesterol 113mg 38%, Sodium 272mg 12%, Total Carbohydrate 32g 12%, Dietary Fiber 1g 3%, Total Sugars 26g, Protein 6g, Vitamin C 0mg 1%, Calcium 89mg 7%, Iron 1mg 5%, Potassium 171mg 4%'}",957
1069,"Looking for an easy baked pie recipe that’s pretty sweet and coconut-y, but not super heavy on calories — any ideas?","Yes — the **Coconut Custard Pie** is the best match for what you want.\n\n- **Easy baked pie:** Yes\n- **Sweet and coconut-y:** Yes, it uses **1 cup flaked coconut** plus vanilla and sugar\n- **Time:** **50 mins**\n- **Servings:** **8**\n- **Calories:** The context doesn’t give calories directly, but it is **not especially light** based on the nutrition facts:\n - **Total Fat:** 24g\n - **Total Sugars:** 43g\n - **Carbohydrate:** 56g\n\nIf you want something **lighter on calories**, the other pie in the context is **Easy Apple Pie**, which has:\n- **Total Fat:** 9g\n- **Total Sugars:** 29g\n\nBut it’s **not coconut-y**. So for your flavor preference, **Coconut Custard Pie** is the right pick.","{'id': '957', 'recipe_name': 'Impossible Coconut Pie', 'total_time': '55 mins', 'servings': '8', 'ingredients': '2 cups milk, ¾ cup white sugar, ½ cup baking mix, ¼ cup unsalted butter, softened, 4 large eggs, 1 ½ teaspoons vanilla extract, 1 cup flaked coconut (Optional)', 'directions': 'Preheat the oven to 350 degrees F (175 degrees C). Grease a 9-inch pie plate. Combine milk, sugar, baking mix, butter, eggs, and vanilla in blender; blend on low speed for 3 minutes. Pour into the prepared pie plate and let stand for 5 minutes. Sprinkle coconut over top. Bake in the preheated oven until a toothpick inserted into the center comes out clean, about 40 minutes. Serve warm.', 'nutrition': 'Total Fat 12g 16%, Saturated Fat 8g 38%, Cholesterol 113mg 38%, Sodium 272mg 12%, Total Carbohydrate 32g 12%, Dietary Fiber 1g 3%, Total Sugars 26g, Protein 6g, Vitamin C 0mg 1%, Calcium 89mg 7%, Iron 1mg 5%, Potassium 171mg 4%'}",957


In [91]:
df_eval.iloc[1069]

question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       Looking for an easy baked pie recipe that’s pretty sweet and coconut-y, but not super heavy on calories — any ideas?
document                                                                                                                                                                                                                                                                                                                                                                            

In [194]:
df_eval.to_csv("data/rag_llm_judge_v3.csv", index=False)